# Phase 232 — Geometry control on the manuscript's own evaluation (Phase 166 hard split, 185-protein test set)

Phase 228 ran the geometry-vs-topology control on the *official* CryptoBench split with CryptoBench's labels.
The manuscript's experimental-site results (Phases 167-189) instead use the **Phase 166 embedding-cluster hard split**
(668 / 89 / 185 graphs) and ligand-contact labels derived from holo structures. Models trained on the official
train split have seen many of those 185 proteins, so Phase 228's checkpoints cannot be scored there. This notebook
retrains the arms on the Phase 166 hard split (identical deterministic recipe) and **exports every test residue
probability** for downstream experimental-site analysis.

Arms (same 51-wide side channel for all): `esm_structure`, `tda` (34), `geom` (17), `geom_tda`, `geom_persistence`.
8 seeds each = 40 trainings. Primary question (fixed in advance): `geom_persistence` vs `geom`.

## Cell 1 — Install

In [ ]:
!pip install -q torch-geometric==2.6.1 scipy scikit-learn

## Cell 2 — Load Phase 159 graphs + Phase 227 geometry, build the Phase 166 hard split

In [ ]:
import copy, json, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef, f1_score, precision_score, recall_score
from scipy.stats import ttest_rel, wilcoxon
from sklearn.cluster import KMeans

def locate(f):
    hits=[]
    for root in (Path('/kaggle/input'), Path('/kaggle/working'), Path('.')):
        if root.exists(): hits.extend(root.rglob(f))
    if not hits: raise FileNotFoundError(f'Could not find {f}')
    return max(hits, key=lambda p:p.stat().st_size)

blob=torch.load(locate('phase159_cryptobench_anmmsf_graphs.pt'),map_location='cpu',weights_only=False)
geo=torch.load(locate('phase227_geometry_features.pt'),map_location='cpu',weights_only=False)
dataset=blob['dataset']
N_PRE=int(blob['n_pre_topo']); TDA_DIM=int(blob['topo_dim']); MSF_COL_IDX=int(blob['msf_col_idx']); POS_WEIGHT=float(blob['pos_weight'])
assert N_PRE==508 and TDA_DIM==34 and MSF_COL_IDX==507
GEOM_NAMES=list(geo['names']); GEOM_DIM=len(GEOM_NAMES); assert GEOM_DIM==17
SIDE_DIM=TDA_DIM+GEOM_DIM   # 51: identical side-channel width for every arm
mu=torch.from_numpy(np.asarray(geo['train_mean'],np.float32)); sd=torch.from_numpy(np.asarray(geo['train_std'],np.float32))

n_bad=0
for d in dataset:
    f=geo['feats'][(str(d.pdb_id).lower(),str(d.chain_id))]
    assert f.shape==(d.x.size(0),GEOM_DIM), (d.pdb_id,f.shape,d.x.shape)
    z=((torch.from_numpy(f)-mu)/sd).clamp(-5,5)          # train-split statistics only
    d.x=torch.cat([d.x,z],dim=1)
assert dataset[0].x.size(1)==N_PRE+SIDE_DIM
OFFICIAL={s:sum(d.split==s for d in dataset) for s in ('train','val','test')}
print('official split:',OFFICIAL)

GPU_OK=False
if torch.cuda.is_available():
    try:
        q=torch.randn(256,256,device='cuda'); (q@q).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print('GPU:',torch.cuda.get_device_name(0),'PASS')
    except Exception as e: print('GPU probe failed:',e)
DEVICE=torch.device('cuda' if GPU_OK else 'cpu')
WORK=Path('/kaggle/working'); (WORK/'cache').mkdir(exist_ok=True)

## Cell 3 — Phase 166 hard split (verbatim recipe) and consistency assertion

In [ ]:
emb=np.stack([d.x[:,27:507].float().mean(0).numpy() for d in dataset])
emb=emb/(np.linalg.norm(emb,axis=1,keepdims=True)+1e-12)
N_CLUSTERS=30
km=KMeans(n_clusters=N_CLUSTERS, random_state=166, n_init=20, max_iter=300)
labels=km.fit_predict(emb)
sizes=np.bincount(labels, minlength=N_CLUSTERS)
targets=np.array([0.70,0.10,0.20])*len(dataset)
rng=np.random.default_rng(166)
best=None
for trial in range(2000):
    order=np.arange(N_CLUSTERS)
    rng.shuffle(order)
    order=sorted(order,key=lambda k:sizes[k],reverse=True)
    counts=np.zeros(3,dtype=int)
    assign={}
    for k in order:
        ratios=(counts+sizes[k])/targets
        j=int(np.argmin(ratios))
        assign[k]=j
        counts[j]+=sizes[k]
    score=float(np.sum(((counts-targets)/targets)**2))
    if best is None or score<best[0]:
        best=(score,counts.copy(),assign.copy())
_,counts,assign=best
hard_split={0:'train_hard',1:'val_hard',2:'test_hard'}
for i,d in enumerate(dataset):
    d.hard_split=hard_split[assign[int(labels[i])]]
sizes_hard={s:sum(d.hard_split==s for d in dataset) for s in hard_split.values()}
print('hard split sizes:',sizes_hard)
# must equal the Phase 166 manifest (668 / 89 / 185) or the split is not the manuscript's
assert sizes_hard=={'train_hard':668,'val_hard':89,'test_hard':185}, sizes_hard
EXPECTED_CLUSTER_ASSIGNMENT={str(k):v for k,v in {0:'train_hard',1:'test_hard',2:'train_hard',3:'val_hard',4:'test_hard',5:'train_hard',6:'train_hard',7:'val_hard',8:'train_hard',9:'train_hard',10:'train_hard',11:'train_hard',12:'train_hard',13:'train_hard',14:'train_hard',15:'test_hard',16:'test_hard',17:'train_hard',18:'train_hard',19:'train_hard',20:'train_hard',21:'train_hard',22:'train_hard',23:'test_hard',24:'val_hard',25:'test_hard',26:'train_hard',27:'train_hard',28:'train_hard',29:'val_hard'}.items()}
got={str(k):hard_split[assign[k]] for k in range(N_CLUSTERS)}
assert got==EXPECTED_CLUSTER_ASSIGNMENT, 'cluster assignment differs from the Phase 166 manifest'
print('cluster assignment identical to phase166_hard_split_manifest.json')
splits={'train':[d for d in dataset if d.hard_split=='train_hard'],'val':[d for d in dataset if d.hard_split=='val_hard'],'test':[d for d in dataset if d.hard_split=='test_hard']}

## Cell 4 — Model (Phase 164/228, unchanged; side channel 51)

In [ ]:
class EGCL(nn.Module):
    def __init__(self,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); self.uc=uc
        self.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        self.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        self.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            self.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(self.c[-1].weight,gain=0.001)
        self.norm=nn.LayerNorm(h)
    def forward(self,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=self.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*self.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=self.norm(h+self.n(torch.cat([h,agg/deg],-1)))
        if self.uc:
            w=self.c(m)*rel/(d2.sqrt()+1.0)
            x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(self,h,t):
        super().__init__(); self.p=nn.Linear(t,h); self.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(self,h,t): return h+self.g(torch.cat([h,t],-1))*self.p(t)

class LocalizingEGNN(nn.Module):
    def __init__(self,n_pre,td,h=128,L=4,dr=0.1):
        super().__init__(); self.n_pre=n_pre
        self.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        self.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        self.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        self.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        self.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))
    def forward(self,d):
        h=self.embed(d.x); pos=d.pos; t=d.x[:,self.n_pre:]
        for l,f in zip(self.ls,self.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits=self.head(h).squeeze(-1)
        batch=d.batch if getattr(d,'batch',None) is not None else torch.zeros(h.size(0),dtype=torch.long,device=h.device)
        loc_logit=self.loc_head(h).squeeze(-1); attn=pyg_softmax(loc_logit,batch)
        pred_center=torch.zeros(int(batch.max().item())+1,3,device=h.device).index_add_(0,batch,attn.unsqueeze(-1)*pos)
        return logits,pred_center

## Cell 5 — Arms

In [ ]:
TOPO_NAMES=list(blob['topo_names']); assert len(TOPO_NAMES)==34
PERSISTENCE_LOCAL=[0,1,2,3,4,5,6, 10,11,12,13,14,15,16,17, 21,22,23,24,25,26,27,28]
TDA_COLS=[N_PRE+i for i in range(34)]
PERS_COLS=[N_PRE+i for i in PERSISTENCE_LOCAL]
GEOM_COLS=[N_PRE+34+i for i in range(GEOM_DIM)]
ARMS={
 'esm_structure':[],
 'tda':TDA_COLS,
 'geom':GEOM_COLS,
 'geom_tda':sorted(GEOM_COLS+TDA_COLS),
 'geom_persistence':sorted(GEOM_COLS+PERS_COLS),
}
def masked_dataset(ds,arm):
    out=[]
    for d in ds:
        d2=copy.copy(d); d2.x=d.x.clone(); keep=torch.zeros(d2.x.size(1),dtype=torch.bool)
        keep[0:507]=True            # structure + ESM (ANM-MSF column 507 stays excluded)
        keep[ARMS[arm]]=True
        d2.x[:,~keep]=0.0
        out.append(d2)
    return out
masked={arm:{s:masked_dataset(splits[s],arm) for s in splits} for arm in ARMS}
for arm in ARMS: print(f'{arm:18s} side-channel features: {len(ARMS[arm])}')

## Cell 6 — Training machinery (unchanged) with test-probability export

In [ ]:
def focal_loss_with_logits(logits,targets,alpha=0.9,gamma=2.0):
    bce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=p*targets+(1-p)*(1-targets); at=alpha*targets+(1-alpha)*(1-targets)
    return (at*(1-pt).pow(gamma)*bce).mean()

def run_epoch(model,loader,opt=None,lam=0.1):
    train=opt is not None; model.train(train); huber=nn.SmoothL1Loss(beta=4.0); ps=[]; ys=[]; errs=[]
    for b in loader:
        b=b.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits,pc=model(b); loss=focal_loss_with_logits(logits,b.y)+lam*huber(pc,b.true_center.view(-1,3))
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy()); errs.append((pc-b.true_center.view(-1,3)).norm(dim=1).detach().cpu().numpy())
    p=np.concatenate(ps); y=np.concatenate(ys); e=np.concatenate(errs); pred=p>=0.5
    return {'pr_auc':float(average_precision_score(y,p)),'roc_auc':float(roc_auc_score(y,p)),'mcc':float(matthews_corrcoef(y,pred)),'f1':float(f1_score(y,pred,zero_division=0)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'dcc4':float((e<4).mean()),'mean_center_error':float(e.mean())}

def train_arm(arm,seed,epochs=40,patience=8,batch=16,lam=0.1):
    torch.manual_seed(seed); np.random.seed(seed)
    model=LocalizingEGNN(N_PRE,SIDE_DIM).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    tl=DataLoader(masked[arm]['train'],batch_size=batch,shuffle=True); vl=DataLoader(masked[arm]['val'],batch_size=batch)
    best=-1; state=None; bad=0
    for ep in range(epochs):
        run_epoch(model,tl,opt,lam); v=run_epoch(model,vl,lam=lam); sch.step()
        if v['pr_auc']>best: best=v['pr_auc']; state={k:t.detach().cpu().clone() for k,t in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: break
    model.load_state_dict(state); model.eval(); val=run_epoch(model,vl,lam=lam); te=run_epoch(model,DataLoader(masked[arm]['test'],batch_size=batch),lam=lam)
    path=WORK/f'cache/phase232_{arm}_seed{seed}.pt'; torch.save(model.state_dict(),path)
    return val,te,str(path)

@torch.no_grad()
def export_test_probs(model,arm):
    model.eval(); out={}
    for d in masked[arm]['test']:
        b=d.clone().to(DEVICE) if hasattr(d,'clone') else d.to(DEVICE)
        logits,_=model(b)
        out[(str(d.pdb_id).lower(),str(d.chain_id))]=torch.sigmoid(logits).cpu().numpy().astype(np.float32)
    return out

## Cell 7 — 5 arms x 8 seeds = 40 trainings

In [ ]:
SEEDS=list(range(8)); results=[]; PROBS={}; t0=time.time()
for arm in ARMS:
    for seed in SEEDS:
        val,te,path=train_arm(arm,seed)
        m=LocalizingEGNN(N_PRE,SIDE_DIM).to(DEVICE); m.load_state_dict(torch.load(path,map_location=DEVICE)); PROBS[(arm,seed)]=export_test_probs(m,arm)
        results.append({'arm':arm,'seed':seed,'val':val,'test':te,'checkpoint':path})
        print(f"{arm:18s} seed={seed} val_PR={val['pr_auc']:.4f} test_PR={te['pr_auc']:.4f} test_ROC={te['roc_auc']:.4f} elapsed={(time.time()-t0)/60:.1f}m")
json.dump(results,open(WORK/'phase232_results.json','w'),indent=2)
torch.save(PROBS,WORK/'phase232_test_probabilities.pt')
print('saved phase232_results.json and phase232_test_probabilities.pt')

## Cell 8 — Summary (protein-level analysis is done locally on the exported probabilities)

In [ ]:
import pandas as pd
rows=[{'arm':r['arm'],'seed':r['seed'],**{f'val_{k}':v for k,v in r['val'].items()},**{f'test_{k}':v for k,v in r['test'].items()}} for r in results]
df=pd.DataFrame(rows); df.to_csv(WORK/'phase232_seed_results.csv',index=False)
print(df.groupby('arm')[['val_pr_auc','test_pr_auc','test_roc_auc']].agg(['mean','std']).round(4).to_string())